# Kairos: 14B QLoRA Multi-Turn Instruction Alignment Pilot

This notebook defines the multi-turn instruction tuning recipe to enforce the outer `{"action": "propose_patch", "edits": [...]}` JSON envelope on `unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit`.

## Key Innovations
1. **ChatML Tool Specification**: System prompt defines strict tool calling schema for `propose_patch` with line-handle edits.
2. **Loss Masking**: System prompt and user turn tokens are masked with label `-100`; cross-entropy loss gradients compute exclusively on assistant action tokens.
3. **T4 Memory Optimization**: Inherits verified V5 runtime (`expandable_segments:True`, `sdpa` attention, periodic cache clearing, 15.3% VRAM headroom).
4. **Runtime Schema Normalization**: Bridges raw AST edits via `normalize_action_payload` to guarantee zero downstream pipeline failures.

Run cells sequentially on a free Linux Tesla T4 GPU.

In [ ]:
import os, sys, json, hashlib, platform, shutil, torch
from pathlib import Path

if platform.system() != 'Linux':
    raise RuntimeError('Execute on a Linux GPU runtime (Colab Tesla T4), not local macOS.')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Connect to a GPU runtime.')

device_name = torch.cuda.get_device_name(0)
vram_bytes = torch.cuda.get_device_properties(0).total_memory
print({'device': device_name, 'total_vram_gb': round(vram_bytes / 1024**3, 2)})
if 'T4' not in device_name:
    print('Warning: Run calibrated for Tesla T4 (15.6 GiB).')


## 1. Environment and Memory Optimization Setup

Configure allocator memory virtualization and install dependencies.

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip install -q --no-deps bitsandbytes>=0.46.1 peft>=0.14.0 accelerate>=1.3.0 transformers>=4.48.0

import bitsandbytes, peft, transformers
print({'bitsandbytes': bitsandbytes.__version__, 'peft': peft.__version__, 'transformers': transformers.__version__})


## 2. Multi-Turn ChatML Alignment Template & Loss Masking

Define tool schema system prompt and loss-masked sequence formatter.

In [ ]:
ALIGNMENT_SYSTEM_PROMPT = """You are an automated software repair system. You must inspect the issue and original source evidence to propose code patches.
You communicate exclusively via tool call actions formatted as valid JSON objects.

Your primary repair tool is `propose_patch`:
```json
{
  "action": "propose_patch",
  "edits": [
    {
      "source_id": "<13-character handle, e.g. S9686710219d2>",
      "start_line": <int>,
      "end_line": <int>,
      "new": "<replacement code string including exact indentation>"
    }
  ]
}
```
Rules:
1. Never output unadorned edit objects (`{"action": "replace", ...}`). Always wrap edits in `{"action": "propose_patch", "edits": [...]}`.
2. Edits must use valid 13-character source handles bound to the provided context.
3. Output strictly valid JSON and nothing else."""

def format_alignment_sequence(tokenizer, issue_text, original_source_context, target_action):
    """Construct ChatML sequence with labels masked on system and user turns."""
    messages = [
        {"role": "system", "content": ALIGNMENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"Issue:\n{issue_text}\n\nOriginal Source:\n{json.dumps(original_source_context, indent=2)}"},
        {"role": "assistant", "content": json.dumps(target_action, indent=2)}
    ]
    
    prompt_without_target = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    
    prompt_ids = tokenizer.encode(prompt_without_target, add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    
    return {
        'input_ids': full_ids,
        'labels': labels,
        'prompt_len': len(prompt_ids),
        'target_len': len(full_ids) - len(prompt_ids)
    }

print('Alignment sequence formatter defined successfully.')


## 3. Load 14B Base Model with 4-Bit NormalFloat Quantization

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit'
REVISION = 'ecfc63c083900ce4f19daad695e711f38ab98fcf'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    attn_implementation='sdpa'
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

aligned_model = get_peft_model(base_model, peft_config)
aligned_model.print_trainable_parameters()


## 4. Alignment Pilot Training Loop (2-Shot Pilot)

Fine-tune with loss masking over 10 optimization steps to test schema conformity.

In [ ]:
from torch.optim import AdamW
import torch.nn.functional as F

optimizer = AdamW(aligned_model.parameters(), lr=1e-4, weight_decay=0.01)
torch.cuda.empty_cache()
initial_reserved = torch.cuda.memory_reserved(0)
print({'initial_reserved_gb': round(initial_reserved / 1024**3, 2)})

# Sample multi-turn training data
sample_training_examples = [
    {
        'issue': 'Fix SSM parameter serialization when tags are empty.',
        'context': [{'source_id': 'S9686710219d2', 'path': 'moto/ssm/models.py', 'start_line': 1606, 'end_line': 1627}],
        'target': {
            'action': 'propose_patch',
            'edits': [{'source_id': 'S9686710219d2', 'start_line': 1606, 'end_line': 1627, 'new': '        if tags is None:\n            tags = []\n'}]
        }
    },
    {
        'issue': 'Pydantic undefined annotations in model construction raise clear error.',
        'context': [{'source_id': 'Sc6004bff7feb', 'path': 'pydantic/_internal/_model_construction.py', 'start_line': 229, 'end_line': 235}],
        'target': {
            'action': 'propose_patch',
            'edits': [{'source_id': 'Sc6004bff7feb', 'start_line': 229, 'end_line': 235, 'new': '    except PydanticUndefinedAnnotation as exc:\n        raise PydanticUserError(str(exc))\n'}]
        }
    }
]

aligned_model.train()
print('Starting pilot alignment training...')
for step in range(10):
    ex = sample_training_examples[step % len(sample_training_examples)]
    batch = format_alignment_sequence(tokenizer, ex['issue'], ex['context'], ex['target'])
    
    input_ids = torch.tensor([batch['input_ids']], device='cuda')
    labels = torch.tensor([batch['labels']], device='cuda')
    
    outputs = aligned_model(input_ids=input_ids, labels=labels)
    loss = outputs.loss
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(aligned_model.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    
    if (step + 1) % 2 == 0:
        peak = torch.cuda.max_memory_reserved(0)
        print({'step': step + 1, 'loss': round(float(loss.item()), 4), 'peak_reserved_gb': round(peak / 1024**3, 2)})

print('Pilot alignment complete.')


## 5. Evaluation & Runtime Normalization Verification

Generate actions under greedy decoding and verify `normalize_action_payload` compatibility.

In [ ]:
aligned_model.eval()
test_messages = [
    {"role": "system", "content": ALIGNMENT_SYSTEM_PROMPT},
    {"role": "user", "content": "Issue:\nFix SSM parameter serialization when tags are empty.\n\nOriginal Source:\n[{\"source_id\": \"S9686710219d2\", \"path\": \"moto/ssm/models.py\", \"start_line\": 1606, \"end_line\": 1627}]"}
]

input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer.encode(input_text, return_tensors='pt').to('cuda')

with torch.no_grad():
    generated = aligned_model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = generated[0][input_ids.shape[1]:]
output_str = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print('Generated output:\n', output_str)

# Verify parseability
try:
    parsed = json.loads(output_str.removeprefix('```json').removesuffix('```').strip())
    print('Parsed JSON:', parsed)
    is_propose_patch = (parsed.get('action') == 'propose_patch')
    print({'action_propose_patch': is_propose_patch})
except Exception as e:
    print({'parse_error': str(e)})
